In [4]:
import polars as pl
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import glob
import os
from datetime import datetime
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
pl.Config.set_tbl_rows(100) 
pl.Config.set_tbl_cols(20)

print('=' * 80)
print('PHÂN TÍCH DỮ LIỆU RECOMMENDATION SYSTEM - POLARS VERSION')
print('=' * 80)

PHÂN TÍCH DỮ LIỆU RECOMMENDATION SYSTEM - POLARS VERSION


In [5]:
print('\nĐANG LOAD DỮ LIỆU VỚI POLARS...')
print('-' * 80)

# Load Items data
print('Loading Items data...')
items_df = pl.read_parquet('sales_pers.item_chunk_0.parquet')
print(f'✓ Items: {items_df.shape[0]:,} rows × {items_df.shape[1]} columns')

# Load Users data
print('\nLoading Users data...')
user_files = sorted(glob.glob('sales_pers.user_chunk_*.parquet'))
users_df = pl.read_parquet(user_files)
print(f'✓ Users: {users_df.shape[0]:,} rows × {users_df.shape[1]} columns')

# Load Purchase History data
print('\nLoading Purchase History data...')
purchase_pattern = 'sales_pers.purchase_history_daily_chunk_*.parquet'
print(f'  Tìm thấy pattern: {purchase_pattern}')

# Sử dụng scan_parquet cho lazy evaluation (tối ưu nhất)
purchases_df = pl.scan_parquet(purchase_pattern).collect()
print(f'✓ Purchases: {purchases_df.shape[0]:,} rows × {purchases_df.shape[1]} columns')

print('\n' + '=' * 80)
print('LOAD DỮ LIỆU HOÀN TẤT!')
print('=' * 80)
print(f'Tổng bộ nhớ sử dụng: ~{(items_df.estimated_size() + users_df.estimated_size() + purchases_df.estimated_size()) / 1e9:.2f} GB')



ĐANG LOAD DỮ LIỆU VỚI POLARS...
--------------------------------------------------------------------------------
Loading Items data...
✓ Items: 27,332 rows × 34 columns

Loading Users data...
✓ Users: 4,573,964 rows × 18 columns

Loading Purchase History data...
  Tìm thấy pattern: sales_pers.purchase_history_daily_chunk_*.parquet
✓ Purchases: 35,729,825 rows × 16 columns

LOAD DỮ LIỆU HOÀN TẤT!
Tổng bộ nhớ sử dụng: ~7.81 GB


# TASK 4: CHUẨN HÓA VÀ BIẾN ĐỔI DỮ LIỆU

#### Mục tiêu: Làm cho dữ liệu có cùng thang đo để giúp mô hình học máy hội tụ nhanh hơn và ổn định hơn.

## Chuẩn hóa dữ liệu số (Min-Max Scaling)

##### Khi các thuộc tính như price, age, income có giá trị chênh lệch nhau lớn, mô hình có thể thiên vị các thuộc tính có thang đo cao hơn. Do đó, ta đưa toàn bộ dữ liệu về cùng một thang [0, 1].

In [6]:
df = users_df.clone()

In [7]:
num_cols = [col for col, dtype in zip(df.columns, df.dtypes) if dtype in [pl.Float64, pl.Int64]]

print(f"Các cột số được chuẩn hóa: {num_cols}")

Các cột số được chuẩn hóa: ['timestamp', 'install_date']


In [8]:
for col in num_cols:
    min_val = df[col].min()
    max_val = df[col].max()
    df = df.with_columns(
        ((pl.col(col) - min_val) / (max_val - min_val)).alias(f"{col}_scaled")
    )

print("✅ Đã chuẩn hóa dữ liệu số bằng Min-Max Scaling.")
print(f"Tổng số dòng dữ liệu hiện có: {df.height}")

✅ Đã chuẩn hóa dữ liệu số bằng Min-Max Scaling.
Tổng số dòng dữ liệu hiện có: 4573964


In [18]:
for col in num_cols:
    scaled_col = f"{col}_scaled"
    min_scaled = df[scaled_col].min()
    max_scaled = df[scaled_col].max()
    print(f"🧮 {scaled_col}: min = {min_scaled:.4f}, max = {max_scaled:.4f}")

🧮 timestamp_scaled: min = 0.0000, max = 1.0000
🧮 install_date_scaled: min = 0.0000, max = 1.0000


In [21]:
cols_to_show = []
for col in num_cols:
    cols_to_show.extend([col, f"{col}_scaled"])

print("\n🔍 So sánh một vài cột trước và sau khi chuẩn hóa:")
display(df.select(cols_to_show).head(15))


🔍 So sánh một vài cột trước và sau khi chuẩn hóa:


timestamp,timestamp_scaled,install_date,install_date_scaled
i64,f64,i64,f64
1306357911,0.0,1306281600,0.0
1306357911,0.0,1306281600,0.0
1312126692,0.012737,1582070400,0.608928
1314302782,0.017542,1314230400,0.017551
1314310024,0.017558,1314230400,0.017551
1314778921,0.018593,1625097600,0.70393
1315308054,0.019761,1315267200,0.01984
1315774658,0.020792,1588377600,0.622854
1315994598,0.021277,1315958400,0.021366


## Biến đổi dữ liệu phân loại (Label Encoding)

In [9]:
cat_cols = [col for col, dtype in zip(df.columns, df.dtypes) if dtype == pl.Utf8]

print(f"Các cột dạng phân loại cần mã hóa: {cat_cols}")

Các cột dạng phân loại cần mã hóa: ['gender', 'province', 'membership', 'sync_error_message', 'region', 'location_name', 'install_app', 'district', 'user_id']


In [15]:
for col in cat_cols:
    df = df.with_columns(
        pl.col(col)
        .cast(pl.Categorical)
        .to_physical()
        .alias(f"{col}_encoded")
    )

print("✅ Đã mã hóa các cột phân loại bằng Label Encoding (Polars).")

✅ Đã mã hóa các cột phân loại bằng Label Encoding (Polars).


In [20]:
df.select([f"{col}_encoded" for col in cat_cols]).head(20)

gender_encoded,province_encoded,membership_encoded,sync_error_message_encoded,region_encoded,location_name_encoded,install_app_encoded,district_encoded,user_id_encoded
u32,u32,u32,u32,u32,u32,u32,u32,u32
1,3,129,null,132,158,1135,1176,5730
0,3,129,null,132,160,1135,1161,5738
0,6,129,null,134,186,1135,6,5745
0,3,129,null,132,187,1135,1188,5750
0,3,129,null,132,188,1135,1151,5754
0,3,129,null,132,189,1135,1161,5759
0,3,129,null,132,190,1135,1228,5766
0,21,129,null,134,191,1135,1183,5772
0,3,129,null,132,192,1135,1232,5778


# TASK 5: RÚT TRÍCH ĐẶC TRƯNG

#### Mục tiêu: Tạo ra những đặc trưng mới có ý nghĩa thống kê hoặc mang giá trị dự báo cáo. Với bài toán dự đoán mua hàng, ta cần tạo ra các đặc trưng phản ánh hành vi mua sắm, giá trị khách hàng, hoặc yếu tố thời gian.

## Tạo đặc trưng tổng chi tiêu (total_spent)

##### Lý do: Khách hàng mua nhiều sản phẩm với số lượng lớn thường có khả năng cao là người tiêu dùng thường xuyên.
##### Do đó, total_spent = quantity × price là đặc trưng rất quan trọng.

In [10]:
if "quantity" in df.columns and "price" in df.columns:
    df = df.with_columns((pl.col("quantity") * pl.col("price")).alias("total_spent"))
    print("✅ Đã thêm đặc trưng total_spent (Tổng số tiền chi tiêu).")
else:
    print("⚠️ Không tìm thấy cột 'quantity' hoặc 'price', bỏ qua bước này.")

⚠️ Không tìm thấy cột 'quantity' hoặc 'price', bỏ qua bước này.


## Phân loại nhóm khách hàng dựa trên tổng chi tiêu

In [12]:
if "total_spent" in df.columns:
    q75 = df["total_spent"].quantile(0.75)
    q50 = df["total_spent"].quantile(0.5)

    df = df.with_columns(
        pl.when(pl.col("total_spent") > q75)
          .then("High Spender")
          .when(pl.col("total_spent") > q50)
          .then("Medium Spender")
          .otherwise("Low Spender")
          .alias("spender_category")
    )
    print("✅ Đã phân loại khách hàng thành các nhóm chi tiêu (High / Medium / Low).")

In [13]:
print("📊 Tổng số cột sau khi tạo đặc trưng:", len(df.columns))
print("Một số cột mới đã được thêm:")
print([col for col in df.columns if "_scaled" in col or "_encoded" in col or col in ["total_spent", "spender_category"]])
df.head()

📊 Tổng số cột sau khi tạo đặc trưng: 20
Một số cột mới đã được thêm:
['timestamp_scaled', 'install_date_scaled']


customer_id,gender,location,province,membership,timestamp,created_date,updated_date,sync_status_id,last_sync_date,sync_error_message,region,location_name,install_app,install_date,district,user_id,is_deleted,timestamp_scaled,install_date_scaled
i32,str,i32,str,str,i64,datetime[μs],datetime[μs],i32,datetime[μs],str,str,str,str,i64,str,str,bool,f64,f64
14732,"""Nam""",155,"""Hồ Chí Minh""","""Standard""",1306357911,2011-05-25 21:11:51.677,2025-07-07 15:33:10.201316,2,2025-07-16 11:54:29.816986,null,"""Đông Nam Bộ""","""HCM - Grand View Phú Mỹ Hưng""","""In-Store""",1306281600,"""7""","""e1e48206652bf8c279ff0206c69a80…",false,0.0,0.0
15126,"""Nữ""",300,"""Hồ Chí Minh""","""Standard""",1306357911,2011-05-25 21:11:51.677,2025-07-07 15:33:10.201316,2,2025-07-16 11:54:29.816986,null,"""Đông Nam Bộ""","""HCM - 121A Nguyễn Duy Trinh""","""In-Store""",1306281600,"""Thủ Đức""","""77891759204bd27e69fb11a7b92889…",false,0.0,0.0
29718,"""Nữ""",157,"""Bến Tre""","""Standard""",1312126692,2011-07-31 15:38:12.750,2025-07-07 15:33:10.201316,2,2025-07-16 11:54:29.816986,null,"""Đồng bằng sông Cửu Long""","""BTR - 179 Nguyễn Đình Chiểu""","""In-Store""",1582070400,"""Bến Tre""","""b8041b584a0bb6655361727a0a6108…",false,0.012737,0.608928
30077,"""Nữ""",53,"""Hồ Chí Minh""","""Standard""",1314302782,2011-08-25 20:06:22.797,2025-07-07 15:33:10.201316,2,2025-07-16 11:54:29.816986,null,"""Đông Nam Bộ""","""HCM - 101 Trần Quang Khải""","""In-Store""",1314230400,"""1""","""c52ceaca44a83ec41a219cfaff0e05…",false,0.017542,0.017551
30085,"""Nữ""",660,"""Hồ Chí Minh""","""Standard""",1314310024,2011-08-25 22:07:04.267,2025-07-07 15:33:10.201316,2,2025-07-16 11:54:29.816986,null,"""Đông Nam Bộ""","""HCM - 85-87 Tây Thạnh""","""In-Store""",1314230400,"""Tân Phú""","""74ac5764f5d67d341aacf238d7762f…",false,0.017558,0.017551
